In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import  Distance, VectorParams, PointStruct
from dotenv import load_dotenv
import os, openai

import openai
import pandas as pd

load_dotenv()  # reads .env in project root
openai.api_key = os.getenv("OPENAI_API_KEY")


In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

qdrant_client.create_collection(
    collection_name="Amazon-items-collections-00",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

In [ ]:
df_items = pd.read_json("../notebook/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [ ]:
df_items.head(2)

In [ ]:
def preprocess_data(row):
    return f"{row['title']} {' '.join(row['description'])}"

In [ ]:
df_items["preprocessed_data"] = df_items.apply(preprocess_data, axis=1)

In [ ]:
df_items.head(2)

In [ ]:
df_sample = df_items.sample(n=50, random_state=42)

In [ ]:
def get_embedding(text, model="text-embedding-ada-002"):
    response = openai.embeddings.create(
        input=[text],
        model=model,
    )   
    return response.data[0].embedding
    

In [ ]:
get_embedding("hi, my name is rahul")

In [ ]:
data_to_embed = df_sample["preprocessed_data"].to_list()
pointstruct_list = []
for i, data in enumerate(data_to_embed):
    pointstruct_list.append(
        PointStruct(id=i, vector=get_embedding(data), payload={"text": data})
    )   

In [ ]:
pointstruct_list

In [ ]:
qdrant_client.upsert(collection_name="Amazon-items-collections-00", wait=True, points=pointstruct_list)

In [ ]:
def retrieve_data(query):
    query_embedding = get_embedding(query)
    results = qdrant_client.search(
        collection_name="Amazon-items-collections-00",
        query_vector=query_embedding,
        limit=5,
    )
    return results

In [ ]:
retrieve_data("WiFi Range Extender").points